# Historical JAXA extraction-tile definitions

This notebook divides each JAXA Vietnam 2020 v23.09 tile's geographic bounds into four extraction regions.
It writes a JSON lookup of bounding boxes; it does not split raster pixels or build the crop-mask mosaic.
The current Python equivalents are [`replication/prepare_inputs.py`](replication/prepare_inputs.py) for bounds and [`replication/prepare_crop_mask.py`](replication/prepare_crop_mask.py) for the categorical mosaic.

## Setup and inputs

Start Jupyter from the repository root with the GDAL Python bindings installed.
Set `input_dir` to the original JAXA tile directory.
The local layout is `datasets/Vietnam_v23.09_10m/`; each source is a north-up, square-cell, WGS84 GeoTIFF.
The original formula uses the x resolution for both axes and assumes no rotation.
The current Python helper validates the corresponding metadata more explicitly.

## Output and conventions

`JAXA_subtiles_full.json` maps each `NxxExxx_i_j` ID to `[west, south, east, north]` in degrees.
`i` increases from west to east and `j` from north to south.
Each region spans half the source extent in both directions; odd raster dimensions do not imply integer-pixel partitions.
No rice-class filtering happens here.
The crop-map extraction later selects categorical class 3 from band `b1`.

Execution counts and the old tile-list output have been cleared.
The code cells were run in a temporary output directory using the original 60 JAXA tiles, with the input path supplied explicitly.
They produced 240 bounding boxes; the temporary output was compared with direct source-raster bounds.
See [`NOTEBOOKS.md`](NOTEBOOKS.md) for the available Python workflow and verification commands.


## 1. Load dependencies and select the input directory

`pathlib` keeps filename parsing independent of Windows or Unix separators.

In [1]:
from pathlib import Path
from osgeo import gdal

input_dir = Path('datasets/Vietnam_v23.09_10m')

## 2. Construct four bounding boxes per source tile

The tile list is sorted for stable dictionary order.
The original geographic arithmetic and west/east then north/south loop order are retained.
The code reads raster metadata without loading the complete tile into memory.

In [2]:
# Getting all available tile paths
raster_list = [str(path) for path in input_dir.glob('*.tif')]
raster_list.sort()

tiles_dict = dict()

# For each tile
for raster_path in raster_list:

    raster = gdal.Open(raster_path)
    gt = raster.GetGeoTransform()

    raster_name = Path(raster_path).name[:7]


    # Getting raster dimensions
    xmin = gt[0]
    ymax = gt[3]
    res = gt[1]
    xlen = res * raster.RasterXSize
    ylen = res * raster.RasterYSize

    # Split each tile into four equal sized subtiles
    div = 2
    xsize = xlen / div
    ysize = ylen / div

    xstep =[xmin + xsize * i for i in range(div+1)]
    ystep =[ymax - ysize * i for i in range(div+1)]

    # Redefining each subtile's bounds
    for i in range(div):
        for j in range(div):
            xmin = xstep[i]
            xmax = xstep[i+1]
            ymax = ystep[j]
            ymin = ystep[j+1]

            # appending new subtile and its bounds to the dict
            tile_name = raster_name+'_'+str(i)+'_'+str(j)
            tiles_dict[tile_name] = [xmin, ymin, xmax, ymax]

## 3. Inspect a bounded preview

The preview shows the first four bounding boxes and the total count; it does not print the full mapping.

In [3]:
print(f'{len(tiles_dict)} subtiles')
for key, bounds in list(tiles_dict.items())[:4]:
    print(key, bounds)

240 subtiles
N08E104_0_0 [104.0, 8.5, 104.5, 9.0]
N08E104_0_1 [104.0, 8.0, 104.5, 8.5]
N08E104_1_0 [104.5, 8.5, 105.0, 9.0]
N08E104_1_1 [104.5, 8.0, 105.0, 8.5]


## 4. Save the tile lookup

This overwrites `JAXA_subtiles_full.json` in the working directory.
The JSON stores extraction extents rather than administrative boundaries or surveyed plots.

In [4]:
# Export to JSON file
import json
with open('JAXA_subtiles_full.json', 'w') as fp:
    json.dump(tiles_dict, fp)